#### Cactus Graph Script

In [3]:
import igraph
import graph_algorithms as g


Pseudocode
1. Find all F-cuts
- 3-cuts
- bridgeless 4-cuts
- corners of crossing bridge-free 4-cuts
3. Get all basic cuts

Function: Iscrossing (cut, cut)

4. Brides of the cactus are nontrivial basic cuts
- Parition the graph
5. Recurse each subgraph until all base cuts are either a cycle or cube
- Shrink each shore into a 'dummy vertex' and recurse

In [36]:
# given: edge list
edges = [[1,5,1],[2,5,1],[1,2,1],[2,4,1],[4,6,1],[5,6,1],[4,5,1],[3,4,1],[3,6,1],[2,3,1],[1,3,1]]
edges =[[1,2,1],[3,4,2],[2,4,2],[1,3,2]]
edges = [[1,5,1],[1,6,1],[4,5,1],[5,6,1],[4,6,1],[4,8,1],[7,8,1],[7,9,1],[8,9,1],[5,7,1],[7,10,1],[2,9,1],[2,3,3],[1,3,1],[3,10,3]]
edges = [[1,3,2],[3,4,2],[2,4,2],[1,2,2]]
vertices = set()
for u, v, w in edges:
    vertices.add(u)
    vertices.add(v)
n = len(vertices)
vertices


{1, 2, 3, 4}

In [37]:
# Verify if the graph is 3ec
cc = g.k_edge_connected_components(edges,3)
print(cc)
if cc == [[]] or len(cc[0][0]) != n:
    print("Graph is not 3 edge connected")

[[]]
Graph is not 3 edge connected


In [38]:
# Get all 3 cuts, 4 cuts 
cuts_3 = g.k_cut(edges, 3)
cuts_4 = g.k_cut(edges, 4)
shores_3 = [shore for cut in cuts_3 for shore in cut]
cuts_3 = [list(map(set, cut)) for cut in g.k_cut(edges, 3)]
cuts_4 = [list(map(set, cut)) for cut in g.k_cut(edges, 4)]
shores_3 = [shore for cut in cuts_3 for shore in cut]



In [39]:
# Get all bridge cuts
bridge_cuts = []
for cut in cuts_4:
    for X in cut: # for every shore
        for Y in shores_3: # for every 3-cut shore
            if Y < X: # if Y \in X
                Y2 = X - Y
                if Y2 in shores_3:
                    bridge_cuts.append(cut)
bridge_cuts = list({frozenset(map(frozenset, cut)) for cut in bridge_cuts}) # remove dupes
bridge_cuts = [list(map(set, cut)) for cut in {frozenset(map(frozenset, cut)) for cut in bridge_cuts}]
bridge_cuts

[]

In [40]:
# Get bridge free cuts
print(bridge_cuts)
print(cuts_4)
cuts_4_static = {frozenset(map(frozenset, cut)) for cut in cuts_4}
bridge_cuts_static = {frozenset(map(frozenset, cut)) for cut in bridge_cuts}

bridgefree_cuts = [list(map(set, cut)) for cut in cuts_4_static - bridge_cuts_static]
bridgefree_cuts

[]
[[{1}, {2, 3, 4}], [{1, 2}, {3, 4}], [{1, 3}, {2, 4}], [{1, 2, 3}, {4}], [{1, 2, 4}, {3}], [{1, 3, 4}, {2}]]


[[{1, 2, 3}, {4}],
 [{2}, {1, 3, 4}],
 [{3, 4}, {1, 2}],
 [{3}, {1, 2, 4}],
 [{2, 4}, {1, 3}],
 [{1}, {2, 3, 4}]]

In [41]:
# Crossign cuts
def isCrossing(cut1, cut2):
    '''
    Given 2 cuts as list(set), returns whether they are crossing and a list of intersections
    '''
    A,B = cut1
    C,D = cut2
    intersections = [A & C, A & D, B & D, B & C]
    flag = True
    for i in intersections:
        if len(i) == 0:
            flag = False
    return flag, intersections

In [42]:
corner_cuts = []
for i in range(len(bridgefree_cuts)):
    for j in range(i, len(bridgefree_cuts)):
        crossing, intersections = isCrossing(bridgefree_cuts[i], bridgefree_cuts[j])
        if (crossing):
            for shore in intersections:
                corner_cuts.append([shore, vertices - shore]) # This is a bit janky but should work
corner_cuts

[[{4}, {1, 2, 3}], [{3}, {1, 2, 4}], [{1}, {2, 3, 4}], [{2}, {1, 3, 4}]]

In [57]:
# Family of F cuts
F_cuts = []
F_cuts.extend(cuts_3)
F_cuts.extend(bridgefree_cuts)
F_cuts.extend(corner_cuts)
F_cuts = [list(map(set, cut)) for cut in {frozenset(map(frozenset, cut)) for cut in F_cuts}]
F_cuts

[[{1, 2, 3}, {4}],
 [{3}, {1, 2, 4}],
 [{2}, {1, 3, 4}],
 [{3, 4}, {1, 2}],
 [{2, 4}, {1, 3}],
 [{1}, {2, 3, 4}]]

In [54]:
# Get basic Fcuts
Basic_F_cuts = []
for cut1 in F_cuts:
    flag = True
    for cut2 in F_cuts:
        if cut1 != cut2 and isCrossing(cut1, cut2)[0]:
            flag = False
    if flag:
        Basic_F_cuts.append(cut1)
Basic_F_cuts

[[{1, 2, 3}, {4}], [{3}, {1, 2, 4}], [{2}, {1, 3, 4}], [{1}, {2, 3, 4}]]

In [ ]:
# Get all nontrivial basic cuts
# for i in range(len(Basic_F_cuts)):
#     cut = Basic_F_cuts[i]
#     shore1, shore2 = cut
#     if len(shore1) == 1 or len(shore2) == 1:
#         Basic_F_cuts.remove(cut)
#         i -= 1
# Basic_F_cuts

[]

In [55]:
class graph:
    def __init__(self, verticies, edges):
        self.verticies = verticies
        self.edges = edges

In [15]:
def edit_cuts(v, cuts, dummy, dummyset):
    '''
    Given current vertices, nontrivial basic f cuts, filter them so shores with the dummy node's set are replaced with just the dummy node
    '''
    new_cuts = []
    for s1,s2 in cuts:
        s1 = s1 & v
        s2 = s2 & v
        if dummyset <= s1: 
            s1.add(dummy)
        elif dummyset <= s2:
            s2.add(dummy)
        if len(s1) > 0 and len(s2) > 0:
            new_cuts.append((s1,s2))
    return new_cuts

In [16]:
vertices
n = len(vertices)
currn = n # gloabl dummy node counter

In [71]:
def cactus(verticies, Fcuts, bFcuts, e):
    '''
    Recursive cactus function that returns a graph struct
    Takes in verticies, basic f-cuts
    '''
    v = verticies
    global currn
    n = len(v)
    # Base Case: Vertex 
    if len(v) == 1:
        return graph(verticies = v, edges = [])

    # Check if there exists a nontrivial basic cut. If there is, then partition, create dummy nodes, and recurse
    for cut in bFcuts:
        shore1, shore2 = cut
        p1 = shore1 & v # p - partitions
        p2 = shore2 & v

        print("CUT", shore1, shore2)
        print("PARTITIONS", p1,p2)
        if len(p1) >= 2 and len(p2) >= 2: # The cut is partitioning the subgraph, so we recurse

            # Add dummy nodes
            a = currn + 1
            b = currn + 2
            currn += 2
            # Partition verticies
            v1 = p1 | {a} 
            v2 = p2 | {b}

            new_Fcuts1 = edit_cuts(v1,Fcuts,a,p2)
            new_Fcuts2 = edit_cuts(v2,Fcuts,b,p1)

            new_bFcuts1 = edit_cuts(v1,bFcuts,a,p2)
            new_bFcuts2 = edit_cuts(v1,bFcuts,b,p1)

            cactus1 = cactus(v1, new_Fcuts1, new_bFcuts1,e)
            cactus2 = cactus(v2,new_Fcuts2, new_bFcuts2,e)

            verticies = cactus1.verticies | cactus2.verticies #union
            edges = cactus1.edges + cactus2.edges + [(a,b)]

            return graph(verticies = verticies, edges = edges)
    # All 3-cuts are trivial, so we do checks:
    
    # Check if the graph is cubic
    degrees = {_v : 0 for _v in v}
    print(degrees)
    for u,v,w in e:
        print(u,v)
        degrees[u] += 1
        degrees[v] += 1
    if all(d == 3 for d in degrees.values()):
        print("graph is cubic!")
        return graph(verticies = verticies, edges = edges)

    # Else there is a cycle
    # Since every cactus node is empty or single it should be safe to treat everythig as a single vertex? and we use F cuts i thnk

    new_edges = []
    corners = None
    flag2 = False
    # 1. Find Crossing cuts
    for i in range(len(Fcuts)):
        for j in range(i,len(Fcuts)):
            flag, corners = isCrossing(Fcuts[i],Fcuts[j])
            # Corners from isCrossing should be in a cycle order
            if flag:
                flag2 = True 
                break 
        if flag2: break
    if not flag:
        raise ValueError("Crossing cuts not found")

    # 2. Get node ordering for the cycle
    for i in range(len(corners)):
        new_edges.append((corners[i], corners[(i+1)%len(corners)]))

    return graph(verticies=verticies, edges=new_edges)



In [70]:
gg = cactus(vertices,F_cuts,Basic_F_cuts,edges)
gg.edges

eee = [list(t) for t in gg.edges]
print(gg.edges)



CUT {1, 2, 3} {4}
PARTITIONS {1, 2, 3} {4}
CUT {3} {1, 2, 4}
PARTITIONS {3} {1, 2, 4}
CUT {2} {1, 3, 4}
PARTITIONS {2} {1, 3, 4}
CUT {1} {2, 3, 4}
PARTITIONS {1} {2, 3, 4}
{1: 0, 2: 0, 3: 0, 4: 0}
1 3
3 4
2 4
1 2
[({4}, {3}), ({3}, {1}), ({1}, {2}), ({2}, {4})]
